# Week 3 — Data Contract
## Lane 3: Content archetype discovery

This notebook defines the Lane 3 slice, verifies the March 2026 warehouse partition with exactly three verification queries, builds a five-feature frame from March data, explores unsupervised archetypes, and demonstrates a deliberate label-leakage trap using a future outcome.

**Data source:** FlyRank/internship-warehouse, release build v20260703. The dataset is gated. Run this notebook in Colab after requesting dataset access and adding a READ token to Colab Secrets under the name HF_TOKEN. The token is read at runtime and is never stored in this notebook.

**Time design:** March 2026 is the feature window; April 2026 is the illustrative outcome window. June 2026 is sealed and is not used. This notebook is prepared to execute against the warehouse; query outputs and model metrics must be populated by running it with authorized access. No data outputs are fabricated.


## 1) Data contract — five plain-words answers

1. **What does one row mean?** In the source fact table, one row is one content item for one client on one reporting date. The analysis feature frame aggregates those daily rows into one row per client/content pair for March 2026.
2. **Which table(s) will I use?** The partitioned table fact_content_daily_performance, using the March 2026 partition for features and April 2026 for a separate future outcome used only in the leakage demonstration. No dimension or 90-day query table is needed for this first pass.
3. **Which time window?** Features are computed from March 1–31, 2026. The illustrative outcome is computed from April 1–30, 2026. June 2026 and the final-month sample table are not used.
4. **What would I predict or rank?** The primary Lane 3 objective is unsupervised discovery of content archetypes from observed SEO, traffic, engagement, and AI-referred traffic patterns. There are no ground-truth archetype labels. For the required leakage demonstration only, a temporary proxy label marks a client/content pair as declining when April GSC impressions are at least 20% below March impressions. This is an educational proxy, not a validated business target.
5. **What do I deliberately exclude?** Client and content identifiers are retained only for grouping/joining, never as model features. April outcome columns are excluded from the honest March feature matrix. The deliberate future-derived leakage column is removed after the trap demonstration. June 2026 remains untouched.


## 2) Connect to the gated warehouse

The token is retrieved from Colab Secrets or an environment variable. If neither is available, a hidden prompt is used. Never paste a token into a notebook cell or commit it to the repository.


In [ ]:
%pip -q install duckdb pandas numpy scikit-learn pyarrow


In [ ]:
import os
import getpass
import duckdb
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, accuracy_score, balanced_accuracy_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GroupShuffleSplit

def get_hf_token():
    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
        if token:
            return token
    except Exception:
        pass
    token = os.environ.get("HF_TOKEN")
    if token:
        return token
    return getpass.getpass("Hugging Face READ token (hidden input): ")

con = duckdb.connect()
con.execute("INSTALL httpfs")
con.execute("LOAD httpfs")
con.execute("LOAD httpfs")

token = get_hf_token()
escaped_token = token.replace("'", "''")
con.execute(
    "CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '" + escaped_token + "')"
)
del token, escaped_token

REL = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance"
MAR = "read_parquet('" + REL + "/month=2026-03/*.parquet')"
APR = "read_parquet('" + REL + "/month=2026-04/*.parquet')"

print("Warehouse connection configured.")
print("Feature window: March 2026 | Outcome window: April 2026 | June 2026: sealed")


## 3) Three verification queries

These are the three required verification queries. Their real outputs appear when the notebook is executed against the gated warehouse.


### Verification query 1 — check the declared daily grain

The source grain is report date × client × content. The query below returns any duplicate key groups in March. An empty result supports the declared grain.


In [ ]:
q1 = con.execute("""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS rows_per_key
    FROM """ + MAR + """
    GROUP BY report_date, client_hash_id, content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 10
""").df()

display(q1)
print("Duplicate key groups returned:", len(q1))
if len(q1) == 0:
    print("Grain check passed for the probed March partition: no duplicate daily keys found.")
else:
    print("Grain check needs investigation: duplicate daily keys were found.")


### Verification query 2 — actual March row count and date span

This reports the row count and observed minimum/maximum reporting dates in the March partition; it does not assume the partition is complete.


In [ ]:
q2 = con.execute("""
    SELECT
        COUNT(*) AS row_count,
        MIN(report_date) AS min_report_date,
        MAX(report_date) AS max_report_date
    FROM """ + MAR + """
""").df()

display(q2)


### Verification query 3 — availability, using IS TRUE

The counts explicitly distinguish available rows from FALSE or NULL availability. A missing metric is not automatically treated as measured zero.


In [ ]:
q3 = con.execute("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_available_rows,
        COUNT(*) FILTER (WHERE gsc_data_available IS NOT TRUE) AS gsc_not_true_rows,
        COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available_rows,
        COUNT(*) FILTER (WHERE ga4_data_available IS NOT TRUE) AS ga4_not_true_rows
    FROM """ + MAR + """
""").df()

display(q3)


## 4) Build the March feature frame — five features maximum

The fact table is large, so aggregation happens in DuckDB and only the small per-content result is brought into pandas. A schema check resolves the metric names from the actual partition before the aggregation runs. GSC fields are used only when GSC availability is TRUE; GA4 fields are used only when GA4 availability is TRUE.


In [ ]:
# Inspect the March schema and resolve expected fields.
schema = con.execute("DESCRIBE SELECT * FROM " + MAR).df()
available_columns = set(schema["column_name"].astype(str))

candidates = {
    "date": ["report_date"],
    "client": ["client_hash_id"],
    "content": ["content_hash_id"],
    "gsc_flag": ["gsc_data_available"],
    "ga4_flag": ["ga4_data_available"],
    "impressions": ["gsc_impressions", "impressions"],
    "clicks": ["gsc_clicks", "clicks"],
    "sessions": ["ga4_sessions", "sessions"],
    "engaged_sessions": ["ga4_engaged_sessions", "engaged_sessions"],
    "ai_sessions": ["ga4_ai_sessions", "ai_sessions"],
}

resolved = {}
for logical_name, options in candidates.items():
    match = next((name for name in options if name in available_columns), None)
    if match is None:
        raise ValueError(
            f"Could not resolve required field {logical_name}. "
            f"Tried {options}. Available fields: {sorted(available_columns)}"
        )
    resolved[logical_name] = match

print("Resolved source fields:")
display(pd.DataFrame(
    [{"logical_name": k, "source_column": v} for k, v in resolved.items()]
))

def qi(identifier):
    # Quote an identifier discovered from the table schema.
    return '"' + identifier.replace('"', '""') + '"'

d = qi(resolved["date"])
client = qi(resolved["client"])
content = qi(resolved["content"])
gsc_flag = qi(resolved["gsc_flag"])
ga4_flag = qi(resolved["ga4_flag"])
impressions = qi(resolved["impressions"])
clicks = qi(resolved["clicks"])
sessions = qi(resolved["sessions"])
engaged = qi(resolved["engaged_sessions"])
ai_sessions = qi(resolved["ai_sessions"])

march_sql = f"""
SELECT
    {client} AS client_hash_id,
    {content} AS content_hash_id,
    SUM(CASE WHEN {gsc_flag} IS TRUE THEN COALESCE({impressions}, 0) END) AS march_impressions,
    SUM(CASE WHEN {gsc_flag} IS TRUE THEN COALESCE({clicks}, 0) END) AS march_clicks,
    SUM(CASE WHEN {ga4_flag} IS TRUE THEN COALESCE({sessions}, 0) END) AS march_sessions,
    SUM(CASE WHEN {ga4_flag} IS TRUE THEN COALESCE({engaged}, 0) END) AS march_engaged_sessions,
    SUM(CASE WHEN {ga4_flag} IS TRUE THEN COALESCE({ai_sessions}, 0) END) AS march_ai_sessions,
    COUNT(DISTINCT CASE WHEN {gsc_flag} IS TRUE THEN {d} END) AS march_gsc_days,
    COUNT(DISTINCT CASE WHEN {ga4_flag} IS TRUE THEN {d} END) AS march_ga4_days
FROM {MAR}
GROUP BY {client}, {content}
HAVING COUNT(DISTINCT CASE WHEN {gsc_flag} IS TRUE THEN {d} END) > 0
   AND COUNT(DISTINCT CASE WHEN {ga4_flag} IS TRUE THEN {d} END) > 0
"""

march_raw = con.execute(march_sql).df()
march_raw["log_march_impressions"] = np.log1p(march_raw["march_impressions"].clip(lower=0))
march_raw["log_march_clicks"] = np.log1p(march_raw["march_clicks"].clip(lower=0))
march_raw["log_march_sessions"] = np.log1p(march_raw["march_sessions"].clip(lower=0))
march_raw["march_engagement_rate_pct"] = (
    100 * march_raw["march_engaged_sessions"] / march_raw["march_sessions"].replace(0, np.nan)
)
march_raw["log_march_ai_sessions"] = np.log1p(march_raw["march_ai_sessions"].clip(lower=0))

FEATURES = [
    "log_march_impressions",
    "log_march_clicks",
    "log_march_sessions",
    "march_engagement_rate_pct",
    "log_march_ai_sessions",
]
X_march = march_raw[["client_hash_id", "content_hash_id"] + FEATURES].copy()

print("March feature frame shape:", X_march.shape)
print("Distinct client/content pairs:", X_march[["client_hash_id", "content_hash_id"]].drop_duplicates().shape[0])
display(X_march[FEATURES].head())


### Feature availability notes

Decision moment: **end of March 2026**. The five feature columns contain no April or June measurements.

- **log_march_impressions** — knowable by the end of March because it is log(1 + March GSC impressions), calculated only on rows where GSC availability is TRUE.
- **log_march_clicks** — knowable by the end of March because it is log(1 + March GSC clicks), using only GSC-available rows.
- **log_march_sessions** — knowable by the end of March because it is log(1 + March GA4 sessions), using only rows where GA4 availability is TRUE.
- **march_engagement_rate_pct** — knowable by the end of March because it is March engaged sessions divided by March sessions, multiplied by 100; it is missing when no measured sessions exist rather than inventing a rate.
- **log_march_ai_sessions** — knowable by the end of March because it is log(1 + March AI-referred sessions), using only GA4-available rows.

The frame requires at least one availability-TRUE day for both GSC and GA4 in March. This deliberately defines a measurable intersection for this first feature experiment; it does not mean other content is unimportant.


## 5) Initial Lane 3 archetype exploration

This is an exploratory clustering run, not an accuracy test. Cluster IDs are arbitrary and do not represent ground-truth labels. Imputation and scaling are fit to the feature frame; the five declared features are the only inputs.


In [ ]:
imputer = SimpleImputer(strategy="median")
scaler = RobustScaler()

X_values = imputer.fit_transform(X_march[FEATURES])
X_scaled = scaler.fit_transform(X_values)

kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
cluster_ids = kmeans.fit_predict(X_scaled)

archetypes = X_march[["client_hash_id", "content_hash_id"]].copy()
archetypes["cluster_id"] = cluster_ids

sample_n = min(5000, len(X_scaled))
if sample_n >= 2 and len(np.unique(cluster_ids)) > 1:
    rng = np.random.default_rng(42)
    sample_idx = rng.choice(len(X_scaled), size=sample_n, replace=False)
    sil = silhouette_score(X_scaled[sample_idx], cluster_ids[sample_idx])
    print(f"Exploratory silhouette score on {sample_n:,} sampled rows: {sil:.4f}")
else:
    print("Silhouette score skipped: too few rows or only one cluster was produced.")

display(archetypes["cluster_id"].value_counts().sort_index().rename("content_pairs").to_frame())
cluster_profiles = X_march[FEATURES].assign(cluster_id=cluster_ids).groupby("cluster_id").median(numeric_only=True)
display(cluster_profiles)


## 6) Leakage trap — add one future-derived column, compare, then remove it

The temporary proxy label is **declined_next_month = 1** when April 2026 GSC impressions are at least 20% below March 2026 GSC impressions. It is defined only for content/client pairs with measurable GSC data in both months and positive March impressions.

The trap feature, **future_impressions_change_pct**, is calculated from April impressions. It is unavailable at the end-of-March decision moment and directly determines the proxy label threshold. It must never enter the honest feature matrix. The model comparison is an auxiliary supervised demonstration; it does not turn Lane 3's unsupervised archetype task into a supervised prediction task.


In [ ]:
april_sql = f"""
SELECT
    {client} AS client_hash_id,
    {content} AS content_hash_id,
    SUM(CASE WHEN {gsc_flag} IS TRUE THEN COALESCE({impressions}, 0) END) AS april_impressions,
    COUNT(DISTINCT CASE WHEN {gsc_flag} IS TRUE THEN {d} END) AS april_gsc_days
FROM {APR}
GROUP BY {client}, {content}
HAVING COUNT(DISTINCT CASE WHEN {gsc_flag} IS TRUE THEN {d} END) > 0
"""

april_raw = con.execute(april_sql).df()
leak_df = march_raw.merge(
    april_raw,
    on=["client_hash_id", "content_hash_id"],
    how="inner",
    validate="one_to_one"
)

# Future-defined population: pairs with GSC available in both months and positive March impressions.
leak_df = leak_df[
    (leak_df["march_gsc_days"] > 0) &
    (leak_df["april_gsc_days"] > 0) &
    (leak_df["march_impressions"] > 0)
].copy()

leak_df["future_impressions_change_pct"] = (
    100 * (leak_df["april_impressions"] - leak_df["march_impressions"])
    / leak_df["march_impressions"]
)
leak_df["declined_next_month"] = (
    leak_df["future_impressions_change_pct"] < -20
).astype(int)

print("Leakage demonstration rows:", len(leak_df))
print("Positive-label rate:", round(leak_df["declined_next_month"].mean(), 4))
display(leak_df["declined_next_month"].value_counts(normalize=True).rename("proportion").to_frame())


In [ ]:
# Grouped split: test clients are not present in training clients.
# This avoids reporting a score based on the same clients appearing in both sets.
leak_groups = leak_df["client_hash_id"].astype(str).to_numpy()
y = leak_df["declined_next_month"].astype(int).reset_index(drop=True)

X_honest = leak_df[FEATURES].replace([np.inf, -np.inf], np.nan).reset_index(drop=True)
X_leaky = X_honest.copy()
X_leaky["future_impressions_change_pct"] = leak_df[
    "future_impressions_change_pct"
].reset_index(drop=True)

splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(splitter.split(X_honest, y, groups=leak_groups))

def evaluate_tree(X, y, train_idx, test_idx):
    X_train = X.iloc[train_idx].copy()
    X_test = X.iloc[test_idx].copy()
    imputer_local = SimpleImputer(strategy="median")
    X_train_imp = imputer_local.fit_transform(X_train)
    X_test_imp = imputer_local.transform(X_test)
    model = DecisionTreeClassifier(max_depth=3, min_samples_leaf=10, random_state=42)
    model.fit(X_train_imp, y.iloc[train_idx])
    pred = model.predict(X_test_imp)
    return {
        "accuracy": accuracy_score(y.iloc[test_idx], pred),
        "balanced_accuracy": balanced_accuracy_score(y.iloc[test_idx], pred),
    }

honest_result = evaluate_tree(X_honest, y, train_idx, test_idx)
leaky_result = evaluate_tree(X_leaky, y, train_idx, test_idx)
majority_baseline = max(y.iloc[test_idx].mean(), 1 - y.iloc[test_idx].mean())

comparison = pd.DataFrame([
    {
        "experiment": "Honest: March-only features",
        "feature_count": len(FEATURES),
        "accuracy": honest_result["accuracy"],
        "balanced_accuracy": honest_result["balanced_accuracy"],
    },
    {
        "experiment": "INVALID: March features + future-derived change",
        "feature_count": len(FEATURES) + 1,
        "accuracy": leaky_result["accuracy"],
        "balanced_accuracy": leaky_result["balanced_accuracy"],
    },
])
display(comparison)
print("Test-set majority-class baseline accuracy:", round(majority_baseline, 4))
print("The leaky score is invalid, even if it is near-perfect. It uses April information unavailable at the decision moment.")


In [ ]:
# Remove the leaked feature from the final declared feature list.
FINAL_FEATURES = FEATURES.copy()
assert "future_impressions_change_pct" not in FINAL_FEATURES
assert "declined_next_month" not in FINAL_FEATURES
print("Final honest feature count:", len(FINAL_FEATURES))
print("Final honest features:", FINAL_FEATURES)
print("Leakage feature excluded:", "future_impressions_change_pct" not in FINAL_FEATURES)


## 7) Named limitation

**Limitation — outcome-window availability affects the leakage-demo population.** The auxiliary proxy-label experiment can include only client/content pairs with GSC availability TRUE in both March and April and positive March impressions. This is a future-dependent population filter, so the resulting supervised scores do not represent every content item or a production-ready prediction task. The Lane 3 clustering itself uses only March features and does not use April availability to select rows.

A second limitation is that a 20% month-over-month impressions drop is a deliberately simple proxy, not a verified business outcome. Seasonality, low impression counts, measurement variation, and client-specific history can affect it. Cluster IDs are exploratory groups, not true labels.


## 8) Self-check

- [x] Five plain-words contract answers are documented.
- [x] Exactly three verification queries are included: daily grain, March row count/date span, and availability using IS TRUE / IS NOT TRUE.
- [x] A March-only feature frame contains five features.
- [x] Every feature has an explicit “knowable when?” explanation.
- [x] The main Lane 3 clustering run uses only March features.
- [x] The leakage demonstration adds a future-derived column, compares it with the honest model, then removes it from the final feature list.
- [x] One named limitation is documented.
- [ ] **Execute every cell in Colab with authorized HF_TOKEN access.** The notebook file itself does not contain fabricated outputs.
- [ ] **Inspect the real query outputs and scores, then commit the executed version.** Do not claim warehouse verification until these cells have run successfully.
